In [1]:
import io
import threading
import time
import traceback

import numpy as np
import ipywidgets as w
from IPython.display import display
from matplotlib.figure import Figure
from pynq import Overlay, MMIO, allocate

ol = Overlay("bits/main.bit")
gen_ip = ol.ip_dict["sinegen_0"]
regs = MMIO(gen_ip["phys_addr"], gen_ip["addr_range"])
dma = ol.axi_dma_0

CTRL, B1, Y1, LEN = 0x0C, 0x10, 0x14, 0x18
S2MM_DMACR, S2MM_DMASR = 0x30, 0x34

FRAC = 13
PKT_STEP = 16
PKT_MIN, PKT_MAX = 16, 4096
N = 256
W0, AMP0 = np.pi / 8, 0.9
HIST = 1 << 18
DMA_TIMEOUT = 1.0

buf = None

In [2]:
hist = np.zeros(HIST, dtype=np.int16)
total = 0
plot_lock = threading.Lock()

view = w.IntSlider(value=512, min=64, max=4096, step=64, description="Window size", continuous_update=False, style={"description_width": "110px"}, layout=w.Layout(width="520px"))
image = w.Image(format="png")

fig = Figure(figsize=(9, 3), dpi=80)
ax = fig.add_subplot()
line, = ax.plot([], [], lw=1.2)
ax.set_xlabel("n")
ax.set_ylabel("Amp")
ax.grid(True, alpha=0.3)
fig.tight_layout()


def refresh():
    with plot_lock:
        n = min(view.value, total)
        y = hist[(total - n + np.arange(n)) % HIST] / 2**FRAC
        line.set_data(np.arange(n), y)
        ax.set_xlim(0, max(n - 1, 1))
        top = max(1.1, 1.1 * np.abs(y).max()) if n else 1.1
        ax.set_ylim(-top, top)
        png = io.BytesIO()
        fig.savefig(png, format="png")
        image.value = png.getvalue()

In [3]:
lock = threading.RLock()
running = False
thread = None
errors = 0

status = w.Label(value="Stopped")
err_box = w.Textarea(rows=4, disabled=True, layout=w.Layout(width="600px", display="none"))


def set_status(text, err=None):
    status.value = text
    err_box.value = err or ""
    err_box.layout.display = "block" if err else "none"

def _coeffs(w_rad, amp):
    b1 = int(round(2 * np.cos(w_rad) * 2**FRAC))
    w_q = np.arccos(b1 / 2**(FRAC + 1))
    y1 = int(np.clip(round(amp * np.sin(w_q) * 2**FRAC), -32768, 32767))
    return b1, y1

b1_cur, y1_cur = _coeffs(W0, AMP0)

def _wait(done, timeout, what):
    t0 = time.monotonic()
    while not done():
        if time.monotonic() - t0 > timeout:
            raise TimeoutError(what)
        time.sleep(0.0005)

def _reset_dma():
    dma.mmio.write(S2MM_DMACR, 0x4)
    _wait(lambda: not dma.mmio.read(S2MM_DMACR) & 0x4, 1.0, "DMA reset")
    dma.recvchannel.start()

def _apply(enable):
    regs.write(CTRL, 0)
    regs.read(CTRL)
    _reset_dma()
    regs.write(B1, b1_cur & 0xFFFF)
    regs.write(Y1, y1_cur & 0xFFFF)
    regs.write(LEN, N)
    if enable:
        regs.write(CTRL, 1)

def _capture_once():
    def finished():
        sr = dma.mmio.read(S2MM_DMASR)
        if sr & 0x70:
            raise RuntimeError(f"DMA error, DMASR=0x{sr:08x}")
        return sr & 0x1000

    dma.mmio.write(S2MM_DMASR, 0x1000)
    dma.recvchannel.transfer(buf)
    _wait(finished, DMA_TIMEOUT, "DMA transfer timeout")
    buf.invalidate()
    return np.array(buf)

def set_coeffs(b1, y1):
    global b1_cur, y1_cur
    with lock:
        b1_cur, y1_cur = int(b1), int(y1)
        _apply(running)

def set_sine(w_rad, amp):
    set_coeffs(*_coeffs(w_rad, amp))

def set_len(n):
    global N, buf
    n = min(max(int(n) // PKT_STEP * PKT_STEP, PKT_MIN), PKT_MAX)
    with lock:
        old, buf = buf, allocate(shape=(n,), dtype=np.int16)
        if old is not None:
            old.freebuffer()
        N = n
        _apply(running)

def reset_gen():
    with lock:
        _apply(running)

reset_dma = reset_gen

def capture_loop():
    global total, running, errors
    try:
        while running:
            try:
                with lock:
                    if not running:
                        break
                    chunk = _capture_once()
            except (TimeoutError, RuntimeError) as e:
                errors += 1
                with lock:
                    _apply(running)
                set_status(f"Running (DMA errors: {errors}, last: {e})")
                time.sleep(0.1)
                continue

            with plot_lock:
                hist[(total + np.arange(len(chunk))) % HIST] = chunk
                total += len(chunk)
            refresh()
            time.sleep(0.05)
    except Exception:
        set_status("Error: capture stopped", traceback.format_exc())
    finally:
        running = False
        try:
            with lock:
                regs.write(CTRL, 0)
        except Exception:
            pass

def start():
    global running, thread, errors
    if running:
        return
    errors = 0
    with lock:
        _apply(True)
        running = True
    set_status("Running")
    thread = threading.Thread(target=capture_loop, daemon=True)
    thread.start()

def stop():
    global running
    running = False
    if thread is not None and thread.is_alive():
        thread.join(timeout=5)
    with lock:
        regs.write(CTRL, 0)
    if status.value.startswith("Running"):
        set_status("Stopped")

In [4]:
def safe(fn, *args):
    try:
        fn(*args)
    except Exception:
        set_status("Error", traceback.format_exc())

style = {"description_width": "110px"}
layout = w.Layout(width="520px")

freq = w.FloatSlider(value=W0, min=0.02, max=3.10, step=0.01, description="Freq", readout_format=".2f", continuous_update=False, style=style, layout=layout)
amp = w.FloatSlider(value=AMP0, min=0.0, max=3.5, step=0.05, description="Amp", readout_format=".2f", continuous_update=False, style=style, layout=layout)
length = w.IntSlider(value=N, min=PKT_MIN, max=PKT_MAX, step=PKT_STEP, description="Packet len", continuous_update=False, style=style, layout=layout)

btn_start = w.Button(description="Start", button_style="success")
btn_stop = w.Button(description="Stop", button_style="danger")
btn_reset = w.Button(description="Reset")

freq.observe(lambda c: safe(set_sine, freq.value, amp.value), "value")
amp.observe(lambda c: safe(set_sine, freq.value, amp.value), "value")
length.observe(lambda c: safe(set_len, c["new"]), "value")
view.observe(lambda c: refresh(), "value")
btn_start.on_click(lambda _: safe(start))
btn_stop.on_click(lambda _: safe(stop))
btn_reset.on_click(lambda _: safe(reset_gen))

safe(set_len, N)
refresh()

display(w.VBox([
    w.HBox([btn_start, btn_stop, btn_reset, status]),
    freq, amp, length, view,
    err_box,
    image,
]))